# The Human branch at hex grain: where do human-caused fires start and burn?

The third hex-grain surface, and the one that completes the **what / where** split the project now
rests on:

| question | grain | product |
| --- | --- | --- |
| **which causes** to target | EPA Level III region-season | [`08_human_cause.ipynb`](08_human_cause.ipynb) — ranked sub-cause composition |
| **where** to site the work | res-5 H3 hex-season | this notebook |

These are different decisions made with different data, not competing versions of one product. A
planner picks the cause from the Level III profile and the location from the hex surface.

**Why the Human branch deserves its own hex treatment.** Notebooks 12 and 13 built the ignition and
burned-area surfaces for Natural fire, where the mitigation argument is about lightning nobody can
prevent. Human ignition is a different process: it tracks roads, settlement and access rather than
weather, so a hex's own history should describe it *better* than it describes lightning — and the
seasonality should be flatter, since people do not confine their ignitions to summer.

Both expectations are tested below, and both hold.

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
import hex_acres as ha
import hex_panel as hp
from config import ProjectConfig

warnings.filterwarnings("ignore")
cfg = ProjectConfig()
DATA = cfg.data
RNG = np.random.default_rng(0)

SEASONS = {0: "DJF", 1: "MAM", 2: "JJA", 3: "SON"}

ign = hp.build_cached(DATA)
acr = ha.build_cached(DATA, verbose=False)
print(f"ignition panel {ign.shape} | acres panel {acr.shape}")

panel: loaded from hex_panel_modelling.parquet
ignition panel (4166910, 29) | acres panel (4166910, 23)


## Human fire is a four-season phenomenon

The first structural difference from Natural, and it changes how the branch must be scored. Natural
ignition is overwhelmingly summer, so notebooks 12 and 13 restrict to JJA. Human fire does not
behave that way — starts peak in **spring** and acres in **summer**, with meaningful activity in
every season.

Scoring the Human branch JJA-only would therefore discard most of the target.

In [2]:
both = ign.merge(acr[["hex_id", "season_idx", "acres_human"]],
                 on=["hex_id", "season_idx"], how="left")

seasonality = pd.DataFrame({
    "starts": both.groupby("season_ord")["starts_human"].sum(),
    "acres": both.groupby("season_ord")["acres_human"].sum(),
})
seasonality.index = [SEASONS[i] for i in seasonality.index]
seasonality["starts_share"] = seasonality["starts"] / seasonality["starts"].sum()
seasonality["acres_share"] = seasonality["acres"] / seasonality["acres"].sum()

print("Human fire by meteorological season (full record)\n")
print(seasonality.to_string(float_format=lambda x: f"{x:,.3f}"))

nat_jja = ign.loc[ign["season_ord"] == 2, "starts_natural"].sum() / ign["starts_natural"].sum()
print(f"\nFor contrast: {nat_jja:.1%} of NATURAL ignitions fall in JJA alone.")

Human fire by meteorological season (full record)

         starts          acres  starts_share  acres_share
DJF 306,736.000  6,644,933.207         0.175        0.106
MAM 668,493.000 20,047,535.158         0.382        0.319
JJA 436,727.000 24,342,764.083         0.250        0.387
SON 338,360.000 11,812,210.653         0.193        0.188

For contrast: 78.1% of NATURAL ignitions fall in JJA alone.


**Finding — human ignition peaks in spring, natural in summer.** Starts are highest in MAM
while acres are highest in JJA: spring fires are numerous but small, summer fires fewer but larger.
Natural ignition, by contrast, is ~62% JJA.

Every result below is therefore reported **per season** rather than pooled or restricted.

## Ignition likelihood: the strongest surface in the project

Same method as [`12_hex_ignition_baselines.ipynb`](12_hex_ignition_baselines.ipynb) — a trailing
mean of the hex's own same-season history, scored by rank against a shuffled control that holds the
predicted values and destroys only the hex-to-hex mapping.

In [3]:
train, test = hp.split(ign, cfg=cfg)
y = ign["starts_human"].to_numpy(float)
pers = ign["pers_human"].to_numpy()

rows = []
for so, name in SEASONS.items():
    m = test & np.isfinite(pers) & (ign["season_ord"] == so).to_numpy()
    rows.append({
        "season": name, "n_test": int(m.sum()),
        "floor": hp.rank_score(y[m], pers[m]),
        "shuffled": hp.rank_score(y[m], RNG.permutation(pers[m])),
    })
m_all = test & np.isfinite(pers)
rows.append({
    "season": "ALL", "n_test": int(m_all.sum()),
    "floor": hp.rank_score(y[m_all], pers[m_all]),
    "shuffled": hp.rank_score(y[m_all], RNG.permutation(pers[m_all])),
})

print("Human ignition likelihood, held out 2010+\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

nat = hp.baseline_scores(DATA)
print("\nNatural, for comparison:")
print(nat[nat["target"] == "starts_natural"].to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

Human ignition likelihood, held out 2010+

season  n_test   floor  shuffled
   DJF  398574 +0.5298   +0.0008
   MAM  398574 +0.5925   +0.0008
   JJA  398574 +0.4879   +0.0012
   SON  398574 +0.4686   -0.0026
   ALL 1594296 +0.5264   -0.0001

Natural, for comparison:
        target       scope  n_test  spearman
starts_natural all seasons 1594296   +0.3435
starts_natural    JJA only  398574   +0.4106


**Finding — human ignition is more predictable than natural, in every season.**

The all-season floor is **+0.53** against natural's +0.34, and spring reaches **+0.59**. Every
shuffled control sits within ±0.003 of zero, so the skill is spatial in the same sense established
in notebook 12.

The gap is mechanistically expected rather than surprising. Human ignition tracks roads, settlement
and access — infrastructure that barely moves across a 29-year record — so a hex's own history is
close to a complete description of it. Natural ignition depends on where lightning happens to strike
in a given season, which history cannot anticipate.

**For the planner this is the most actionable surface the project produces**: it is the branch where
prevention (rather than mitigation) is the lever, and it is the branch history predicts best.

## Burned area: occurrence and magnitude

The same hurdle split as [`13_hex_acres_baselines.ipynb`](13_hex_acres_baselines.ipynb) — *does this
hex burn at all*, then *how much given that it burns* — with the magnitude baseline conditioned on
burning seasons only.

In [4]:
occ_rows, mag_rows = [], []
y_occ = acr["burned_human"].to_numpy(float)
p_occ = acr["pers_log_human"].to_numpy()
test_a = (acr["season_year"] >= cfg.test_start).to_numpy()

for so, name in SEASONS.items():
    m = test_a & np.isfinite(p_occ) & (acr["season_ord"] == so).to_numpy()
    occ_rows.append({"season": name, "n_test": int(m.sum()),
                     "floor": ha.rank_score(y_occ[m], p_occ[m])})

    _, mag = ha.hurdle_frames(acr, "human", cfg=cfg, season_ord=so)
    mag = mag[mag["season_year"] >= cfg.test_start]
    yy = mag["log_human"].to_numpy()
    pp = mag["persburn_log_human"].to_numpy()
    mag_rows.append({"season": name, "n_test": len(mag),
                     "floor": ha.rank_score(yy, pp),
                     "shuffled": ha.shuffled_null(yy, pp, rng=RNG)})

print("Occurrence — does this hex burn at all?\n")
print(pd.DataFrame(occ_rows).to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
print("\nMagnitude — how much, given it burns?\n")
print(pd.DataFrame(mag_rows).to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

Occurrence — does this hex burn at all?

season  n_test   floor
   DJF  398574 +0.5204
   MAM  398574 +0.5708
   JJA  398574 +0.4683
   SON  398574 +0.4541

Magnitude — how much, given it burns?

season  n_test   floor  shuffled
   DJF   39758 +0.4308   +0.0047
   MAM   77025 +0.4349   +0.0028
   JJA   57266 +0.4464   +0.0063
   SON   50842 +0.4251   +0.0034


**Finding — both stages carry real skill, in every season.** Occurrence runs +0.45 to +0.57 and
magnitude +0.43 to +0.45, with every shuffled control within ±0.003 of zero.

Magnitude is notably *steadier* across seasons than occurrence, and both are close to the natural
branch's magnitude floor (+0.37).

## The tail failure is two orders of magnitude milder than Natural's

This is the result that most distinguishes the two branches, and it is the one that matters for
whether a planner can act on the surface.

Notebook 13 found the natural-acres baseline over-predicting small cells and **under-predicting the
top decile by 270x** across all JJA burning cells, on a median cell of 2,970 acres. (The 855x figure
quoted there is the six-forest-ecoregion subset used for the covariate ladder — a different
population.) The same breakdown on
human acres follows the same monotonic shape — and lands somewhere completely different.

In [5]:
_, mag = ha.hurdle_frames(acr, "human", cfg=cfg, season_ord=None)
mag = mag[mag["season_year"] >= cfg.test_start].copy()
mag["log_err"] = mag["log_human"] - mag["persburn_log_human"]
mag["decile"] = pd.qcut(mag["acres_human"].rank(method="first"), 10, labels=range(1, 11))

by_dec = mag.groupby("decile", observed=True).agg(
    median_acres=("acres_human", "median"),
    median_log_err=("log_err", "median"),
    n=("log_err", "size"),
)
by_dec["x_off"] = 10 ** by_dec["median_log_err"].abs()
by_dec["direction"] = np.where(by_dec["median_log_err"] > 0, "UNDER-predicts", "over-predicts")

print("Human magnitude error by burned-area decile (all seasons, held-out years)\n")
print(by_dec.to_string(float_format=lambda x: f"{x:.2f}"))
print(f"\nTop decile: {by_dec['x_off'].iloc[-1]:.1f}x under, median {by_dec['median_acres'].iloc[-1]:,.0f} acres")
print("Natural, for comparison: 855x under, median 5,073 acres")

Human magnitude error by burned-area decile (all seasons, held-out years)

        median_acres  median_log_err      n  x_off       direction
decile                                                            
1               0.10           -1.00  22490   9.96   over-predicts
2               0.20           -0.73  22489   5.38   over-predicts
3               0.44           -0.52  22489   3.33   over-predicts
4               1.00           -0.43  22489   2.72   over-predicts
5               1.70           -0.22  22489   1.64   over-predicts
6               3.00           -0.03  22489   1.08   over-predicts
7               5.25            0.11  22489   1.29  UNDER-predicts
8              11.00            0.31  22489   2.02  UNDER-predicts
9              27.00            0.58  22489   3.80  UNDER-predicts
10            135.00            1.09  22489  12.31  UNDER-predicts

Top decile: 12.3x under, median 135 acres
Natural, for comparison: 855x under, median 5,073 acres


**Finding — human fire is not only more predictable, its extremes are less extreme.**

The shape is identical to Natural — over-prediction on small cells, crossover near the middle,
under-prediction on large ones — but the magnitude is not. The human top decile is **12.3x** under,
against natural's **270x** on the comparable all-region population. Its median cell is
**135 acres**, against natural's **2,970**.

**Why this matters for the recommendation.** Notebook 13's conclusion was that megafire size is not
forecastable pre-season, so mitigation should be sited against *ignition* rather than against
predicted burn size. That argument was built on the natural branch, where the tail failure is
catastrophic.

It does not transfer wholesale to the human branch. Human burned area is predictable to within about
an order of magnitude even in the top decile, which means a human-branch product can reasonably rank
hexes by *expected acres*, not only by ignition likelihood. The two branches support different
products because their tails behave differently.

## Covariates: a fourth null, and an expected one

The climate and vegetation covariates are run against human ignition for completeness. The prior is
strongly against them: human ignition is driven by settlement geography, and pre-season fuel state
has no mechanism to move where roads and houses are.

Restricted to the six forest ecoregions the MODIS probe covers.

In [6]:
ndvi = pd.read_parquet(DATA / "hex_season_ndvi.parquet")[["hex_id", "season_idx", "ndvi", "evi"]]
six = ign[ign["hex_id"].isin(set(ndvi["hex_id"]))].merge(
    ndvi, on=["hex_id", "season_idx"], how="left")
six = hp.add_climate_anomalies(six, cfg=cfg)

tr_years = six["season_year"] < cfg.test_start
norm = six[tr_years].groupby("hex_id")[["ndvi", "evi"]].mean()
for col in ["ndvi", "evi"]:
    six[f"{col}_anom"] = six[col] - six["hex_id"].map(norm[col])
six = six.sort_values(list(hp.HEX_SORT_KEYS)).reset_index(drop=True)

rungs = {
    "+ climate": list(hp.CLIMATE_COVS),
    "+ NDVI": ["ndvi", "evi"],
    "+ NDVI + climate": ["ndvi", "evi", "ndvi_anom", "evi_anom"] + list(hp.CLIMATE_COVS),
}
print("Human ignition ladder — JJA, six forest ecoregions, held out 2010+\n")
print(hp.ladder(six, "starts_human", rungs, cfg=cfg, season_ord=2)
      .to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

Human ignition ladder — JJA, six forest ecoregions, held out 2010+

                   rung  n_test  spearman  delta_vs_floor
persistence k=7 (floor)   29293   +0.4863         +0.0000
              + climate   29293   +0.4680         -0.0182
                 + NDVI   29293   +0.4818         -0.0045
       + NDVI + climate   29293   +0.4807         -0.0055


**Finding — nothing beats persistence, as expected.** Climate costs −0.018, NDVI −0.005, and
the combination −0.006.

Unlike the natural-branch nulls, this one required no explanation hunting. Pre-season dryness and
vegetation density describe *fuel*, and fuel is not what puts a person next to an ignition source.
The covariates have no mechanism here, and the ladder confirms it.

Counting across the project, this is the fifth covariate result on an ignition target and the fifth
null. The consistent reading stands: **where fires start, at this grain, is a property of the place
rather than of the year** — and for human fire the property is infrastructure rather than terrain.

## Summary

**For an analyst.**

1. **Human ignition is the most predictable surface in the project** — all-season Spearman **+0.53**,
   spring **+0.59**, against shuffled controls within ±0.003 of zero. It beats the natural ignition
   floor (+0.34) in every season.
2. **Human fire is a four-season phenomenon.** Starts peak in MAM and acres in JJA, against ~62% of
   natural ignitions falling in JJA alone, so the branch must be scored per season rather than
   restricted to summer.
3. **Both burned-area stages carry skill**: occurrence +0.45 to +0.57, magnitude +0.43 to +0.45.
4. **The tail failure is far milder than Natural's** — 12.3x under-prediction on the top decile
   against 270x, on a decile whose median cell is 135 acres against 2,970. Both figures are
   all-region, so the comparison is like-for-like.
5. **Covariates add nothing**, and here the null needs no explanation: fuel state has no mechanism
   to move where roads and settlement are.

**In plain language.**

Human-caused fires are easier to predict than lightning fires, and for a simple reason: people start
fires where the roads and houses are, and those do not move. A patch's own history is close to a
full description of its human fire risk — good enough to rank patches within a region far better
than chance, in all four seasons rather than just summer.

They are also less explosive. The largest lightning fires are wildly under-predicted by history; the
largest human fires are under-predicted by about twelve-fold, on cells that burn 135 acres rather
than 5,000. That difference changes what can be built: for human fire it is reasonable to rank
places by *how much is likely to burn*, where for lightning fire only *where it is likely to start*
holds up.

**Where this sits in the product.** This notebook answers **where**. Which human causes to target —
equipment, debris burning, arson, recreation — is a different question answered at EPA Level III
grain in [`08_human_cause.ipynb`](08_human_cause.ipynb). A planner uses the Level III profile to
choose the cause and this surface to choose the location.

**Open.** The Unknown branch has a Level III triage ([`09_unknown_dataquality.ipynb`](09_unknown_dataquality.ipynb))
and no hex-grain counterpart. That asymmetry is defensible — reporting quality is an
agency-and-region property rather than a spatial one — but it is a deliberate choice and should be
stated as such rather than left as an artifact of what happened to get built.